In [21]:
import pandas as pd 
import plotly.express as px
import random 


#definindo bicho
bicho = {
    "data_nascimento":0,
    "vivo":True,
    "expectativa_de_vida":3,
    "data_morte":None,
    "dieta":None 
    
}


#definindo floresta
quantidade_bichos_inicial = 10
lista_de_bichos = []
dieta_lista = ["herbivoro","carnivoro"]

for _ in range(quantidade_bichos_inicial):
    bicho_inicial = bicho.copy()
    dieta = random.choices(dieta_lista,weights=[1,1])[0]
    bicho_inicial["dieta"] = dieta
    lista_de_bichos.append(bicho_inicial)

df_estado_inicial = pd.DataFrame(lista_de_bichos)
df_inicial_dieta = df_estado_inicial.groupby("dieta")["vivo"].count()

print("distribuicao inicial dieta:")
print(df_inicial_dieta)

#loop dos anos
max_anos = 20

for ano_atual in range(1,max_anos+1):
    novos_bichos = []
    for bicho in lista_de_bichos:
        vivo = bicho["vivo"]


        if vivo:
            idade = ano_atual - bicho["data_nascimento"]

            anos_de_vida = round(random.gauss(mu=bicho["expectativa_de_vida"], sigma=1.0))
            if idade >= anos_de_vida:
                bicho['vivo'] = False
                bicho['data_morte'] = ano_atual
            else:
                dieta = bicho["dieta"]
                if dieta == "carnivoro":
                    presa = random.choice(lista_de_bichos)
                    vivo_presa = presa["vivo"]
                    
                    while vivo_presa == False:
                        presa = random.choice(lista_de_bichos)
                        vivo_presa = presa["vivo"]

                    dieta_presa = presa["dieta"]

                    if dieta_presa == "carnivoro":
                        continue
                    else:
                        presa["vivo"] = False
                        presa["data_morte"] = ano_atual
                        novo_bicho = bicho.copy()
                        novo_bicho["data_nascimento"] = ano_atual
                        novos_bichos.append(novo_bicho)
                else:  # é herbivoro   
                    novo_bicho = bicho.copy()
                    novo_bicho["data_nascimento"] = ano_atual
                    novos_bichos.append(novo_bicho)
    lista_de_bichos.extend(novos_bichos)

df = pd.DataFrame(lista_de_bichos)

#adicionando coluna de idade 

df["idade"] = df["data_morte"] - df["data_nascimento"]
df.loc[df["idade"].isna(),"idade"] = max_anos - df.loc[df["idade"].isna(),"data_nascimento"] 

#analises 
def contagem_total_grafico(df):

    # conta quantos bichos nasceram em cada data
    contagem = (
        df
        .groupby("data_nascimento")
        .size()
        .reset_index(name="qtd_bichos")          # renomeia a coluna de contagem
        .sort_values("data_nascimento")         # garante ordenação
    )

    # soma acumulada
    contagem["total_acumulado"] = contagem["qtd_bichos"].cumsum()

    fig = px.line(
        contagem,
        x="data_nascimento",
        y="total_acumulado",
        title="contagem_de_bixos"
    )

    fig.show()



df["idade"].describe()

distribuicao inicial dieta:
dieta
carnivoro    5
herbivoro    5
Name: vivo, dtype: int64


count    29087.000000
mean         1.368756
std          1.201988
min          0.000000
25%          0.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: idade, dtype: float64

In [4]:
df_vivos = df.loc[df["data_morte"].isna()].copy()


resumo_idade_total = df["idade"].describe().to_frame("total")
resumo_idade_herbivoro = df.loc[df["dieta"] == "herbivoro", "idade"].describe().to_frame("herbivoro")
resumo_idade_carnivoro = df.loc[df["dieta"] == "carnivoro", "idade"].describe().to_frame("carnivoro")

resumo_idade_final = pd.concat([resumo_idade_total, resumo_idade_herbivoro, resumo_idade_carnivoro],axis=1)


resumo_idade_total_vivo = df_vivos["idade"].describe().to_frame("total_vivo")
resumo_idade_herbivoro_vivo = df_vivos.loc[df_vivos["dieta"] == "herbivoro", "idade"].describe().to_frame("herbivoro_vivo")
resumo_idade_carnivoro_vivo = df_vivos.loc[df_vivos["dieta"] == "carnivoro", "idade"].describe().to_frame("carnivoro_vivo")

resumo_idade_final_vivo = pd.concat([resumo_idade_total_vivo, resumo_idade_herbivoro_vivo, resumo_idade_carnivoro_vivo],axis=1)

resumo_idade = pd.concat([resumo_idade_final,resumo_idade_final_vivo],axis=1)
resumo_idade

,total,herbivoro,carnivoro,total_vivo,herbivoro_vivo,carnivoro_vivo
count,867.000000,652.000000,215.000000,271.000000,193.000000,78.000000
mean,1.697809,1.575153,2.069767,0.726937,0.658031,0.897436
std,1.123354,1.072858,1.191782,0.811476,0.782058,0.861737
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000
50%,2.000000,1.000000,2.000000,1.000000,0.000000,1.000000
75%,3.000000,2.000000,3.000000,1.000000,1.000000,1.000000
max,4.000000,4.000000,4.000000,3.000000,3.000000,3.000000


In [ ]:
def populacao_ano(df):


    nascimento_ano = df.groupby(["data_nascimento","dieta"]).agg(nascimentos=("dieta","count")).reset_index()
    morte_ano = df.groupby(["data_morte","dieta"]).agg(mortes=("dieta","count")).reset_index()
    df_censo = pd.merge(nascimento_ano,morte_ano,how="outer",left_on=["data_nascimento","dieta"],right_on=["data_morte","dieta"])

    df_censo = df_censo.rename(columns={"data_nascimento":"ano"})
    df_censo.loc[df_censo["ano"].isna(),"ano"] = df_censo.loc[df_censo["ano"].isna(),"data_morte"] 

    df_censo.drop(columns=["data_morte"],inplace=True)

    start_year = int(df_censo['ano'].min())
    end_year = int(df_censo['ano'].max())

    all_years = range(start_year, end_year + 1)
    all_categories = df_censo['dieta'].unique()

    new_index = pd.MultiIndex.from_product(
        [all_categories, all_years], 
        names=['dieta', 'ano']
    )

    df_censo = (
        df_censo.set_index(['dieta', 'ano'])
        .reindex(new_index)
        .reset_index()
    )
    df_censo = df_censo.fillna(0,inplace=False)
    return df_censo 


    


populacao_ano(df)


,dieta,ano,nascimentos,mortes
0,carnivoro,0,0.0,0.0
1,carnivoro,1,0.0,0.0
2,carnivoro,2,0.0,0.0
3,carnivoro,3,0.0,0.0
4,carnivoro,4,0.0,0.0
5,carnivoro,5,0.0,0.0
6,carnivoro,6,0.0,0.0
7,carnivoro,7,0.0,0.0
8,carnivoro,8,0.0,0.0
9,carnivoro,9,0.0,0.0


In [22]:

df.loc[df["idade"]<0]

,data_nascimento,vivo,expectativa_de_vida,data_morte,dieta,idade
